In [33]:
# Ubuntu environment only
# !uv pip install fastapi uvicorn 

### test_api.ipynb

Test notebook for the `api_server.py` FastAPI server running at `http://127.0.0.1:3000`.

**Before running this notebook**, start the server in a terminal:
```bash
python api_server.py
```

#### Endpoint
| Method | URL | Description |
|---|---|---|
| GET | `/` | Health check |
| POST | `/query` | Execute a SELECT SQL query |

#### Request format
```json
{ "sql": "SELECT * FROM suppliers LIMIT 5" }
```

#### Response format (success)
```json
{
  "status": "ok",
  "rows_returned": 5,
  "columns": ["supplier_id", "supplier_name", ...],
  "data": [ { "supplier_id": 1, "supplier_name": "...", ... }, ... ]
}
```

#### Response format (rejected / error)
```json
{ "status": "rejected" | "error", "message": "<reason>" }
```

In [6]:
# !uv pip install requests

In [35]:
import requests
import pandas as pd
import json

BASE_URL = "http://127.0.0.1:3000"

def query(sql: str, label: str = "") -> pd.DataFrame | dict:
    """
    Helper: POST a SQL string to /query and pretty-print the result.
    - On success (status=ok): returns a pandas DataFrame.
    - On rejection/error: prints the message and returns the raw dict.
    """
    if label:
        print(f"\n{'='*60}")
        print(f"  {label}")
        print(f"{'='*60}")
    
    resp = requests.post(f"{BASE_URL}/query", json={"sql": sql})
    body = resp.json()
    
    print(f"HTTP {resp.status_code} | status: {body.get('status')}")
    
    if body.get("status") == "ok":
        df = pd.DataFrame(body["data"], columns=body["columns"])
        print(f"Rows returned: {body['rows_returned']}")
        display(df)
        return df
    else:
        print(f"Message: {body.get('message')}")
        return body

### 1. Health Check

In [39]:
# GET / — confirm server is up
resp = requests.get(f"{BASE_URL}/")
print(f"HTTP {resp.status_code}")
print(resp.json())

HTTP 200
{'status': 'ok', 'message': 'database.db Query API is running. POST /query to execute a SELECT statement.', 'database': '/root/local_jupyter/superai-2026/backend/init_db/database.db'}


### 2. Valid SELECT Queries

In [41]:
# Row counts across all tables
query("""
    SELECT 'suppliers'  AS tbl, COUNT(*) AS rows FROM suppliers  UNION ALL
    SELECT 'products',          COUNT(*)          FROM products   UNION ALL
    SELECT 'inventory',         COUNT(*)          FROM inventory  UNION ALL
    SELECT 'orders',            COUNT(*)          FROM orders     UNION ALL
    SELECT 'sales_items',       COUNT(*)          FROM sales_items
""", label="Row counts per table")


  Row counts per table
HTTP 200 | status: ok
Rows returned: 5


,tbl,rows
0,suppliers,10
1,products,150
2,inventory,150
3,orders,1500
4,sales_items,19862


,tbl,rows
0,suppliers,10
1,products,150
2,inventory,150
3,orders,1500
4,sales_items,19862


In [42]:
# Fetch all suppliers
query("SELECT * FROM suppliers", label="All suppliers")


  All suppliers
HTTP 200 | status: ok
Rows returned: 10


,supplier_id,supplier_code,supplier_name,contact_name,contact_email,contact_phone,address,is_active,created_at
0,1,LIONCI-001,Lion City Foods Pte Ltd,Lion Contact,info@lioncityfoods.example.com,+65 60010001,"11 Market Street, Singapore",1,2024-04-12T00:00:00Z
1,2,EVERGR-002,Evergreen Produce Importers,Evergreen Contact,info@evergreenproduce.example.com,+65 60020002,"12 Market Street, Singapore",1,2024-12-14T00:00:00Z
2,3,PACIFI-003,Pacific Beverages & Co.,Pacific Contact,info@pacificbeverages.example.com,+65 60030003,"13 Market Street, Singapore",1,2024-09-27T00:00:00Z
3,4,GOLDEN-004,Golden Harvest Grocers,Golden Contact,info@goldenharvest.example.com,+65 60040004,"14 Market Street, Singapore",1,2024-04-16T00:00:00Z
4,5,SUNRIS-005,Sunrise Dairy Supplies,Sunrise Contact,info@sunrisedairy.example.com,+65 60050005,"15 Market Street, Singapore",1,2024-03-12T00:00:00Z
5,6,BLUEOC-006,Blue Ocean Seafood Traders,Blue Contact,info@blueoceanseafood.example.com,+65 60060006,"16 Market Street, Singapore",1,2024-07-07T00:00:00Z
6,7,PRIMEC-007,Prime Choice Meats,Prime Contact,info@primechoicemeats.example.com,+65 60070007,"17 Market Street, Singapore",1,2024-01-21T00:00:00Z
7,8,CLEANH-008,CleanHome Household Products,CleanHome Contact,info@cleanhome.example.com,+65 60080008,"18 Market Street, Singapore",1,2024-04-12T00:00:00Z
8,9,FRESHC-009,Fresh & Crisp Snacks,Fresh Contact,info@freshcrispsnacks.example.com,+65 60090009,"19 Market Street, Singapore",1,2024-05-01T00:00:00Z
9,10,HARMON-010,Harmony Personal Care,Harmony Contact,info@harmonycare.example.com,+65 60100010,"20 Market Street, Singapore",1,2024-08-02T00:00:00Z


,supplier_id,supplier_code,supplier_name,contact_name,contact_email,contact_phone,address,is_active,created_at
0,1,LIONCI-001,Lion City Foods Pte Ltd,Lion Contact,info@lioncityfoods.example.com,+65 60010001,"11 Market Street, Singapore",1,2024-04-12T00:00:00Z
1,2,EVERGR-002,Evergreen Produce Importers,Evergreen Contact,info@evergreenproduce.example.com,+65 60020002,"12 Market Street, Singapore",1,2024-12-14T00:00:00Z
2,3,PACIFI-003,Pacific Beverages & Co.,Pacific Contact,info@pacificbeverages.example.com,+65 60030003,"13 Market Street, Singapore",1,2024-09-27T00:00:00Z
3,4,GOLDEN-004,Golden Harvest Grocers,Golden Contact,info@goldenharvest.example.com,+65 60040004,"14 Market Street, Singapore",1,2024-04-16T00:00:00Z
4,5,SUNRIS-005,Sunrise Dairy Supplies,Sunrise Contact,info@sunrisedairy.example.com,+65 60050005,"15 Market Street, Singapore",1,2024-03-12T00:00:00Z
5,6,BLUEOC-006,Blue Ocean Seafood Traders,Blue Contact,info@blueoceanseafood.example.com,+65 60060006,"16 Market Street, Singapore",1,2024-07-07T00:00:00Z
6,7,PRIMEC-007,Prime Choice Meats,Prime Contact,info@primechoicemeats.example.com,+65 60070007,"17 Market Street, Singapore",1,2024-01-21T00:00:00Z
7,8,CLEANH-008,CleanHome Household Products,CleanHome Contact,info@cleanhome.example.com,+65 60080008,"18 Market Street, Singapore",1,2024-04-12T00:00:00Z
8,9,FRESHC-009,Fresh & Crisp Snacks,Fresh Contact,info@freshcrispsnacks.example.com,+65 60090009,"19 Market Street, Singapore",1,2024-05-01T00:00:00Z
9,10,HARMON-010,Harmony Personal Care,Harmony Contact,info@harmonycare.example.com,+65 60100010,"20 Market Street, Singapore",1,2024-08-02T00:00:00Z


In [17]:
# Products grouped by category
query("""
    SELECT category, COUNT(*) AS product_count
    FROM products
    GROUP BY category
    ORDER BY product_count DESC
""", label="Products by category")


  Products by category
HTTP 200 | status: ok
Rows returned: 10


,category,product_count
0,Dairy,19
1,Personal Care,18
2,Frozen,17
3,Beverages,16
4,Seafood,15
5,Snacks,14
6,Meat,14
7,Household,13
8,Bakery,13
9,Produce,11


,category,product_count
0,Dairy,19
1,Personal Care,18
2,Frozen,17
3,Beverages,16
4,Seafood,15
5,Snacks,14
6,Meat,14
7,Household,13
8,Bakery,13
9,Produce,11


In [18]:
# Top 5 products by revenue
query("""
    SELECT p.product_name,
           ROUND(SUM(si.line_sale_amount), 2) AS total_revenue,
           SUM(si.quantity) AS total_units_sold
    FROM sales_items si
    JOIN products p ON si.product_id = p.product_id
    GROUP BY si.product_id
    ORDER BY total_revenue DESC
    LIMIT 5
""", label="Top 5 products by revenue")


  Top 5 products by revenue
HTTP 200 | status: ok
Rows returned: 5


,product_name,total_revenue,total_units_sold
0,CitrusBurst Orange Juice,20464.90,491.0
1,ThirstQuench Apple Juice,17303.77,415.0
2,HappySnacks Potato Chips,16822.40,519.0
3,ThirstQuench Cola,16233.03,448.0
4,CitrusBurst Apple Juice,15513.22,529.0


,product_name,total_revenue,total_units_sold
0,CitrusBurst Orange Juice,20464.90,491.0
1,ThirstQuench Apple Juice,17303.77,415.0
2,HappySnacks Potato Chips,16822.40,519.0
3,ThirstQuench Cola,16233.03,448.0
4,CitrusBurst Apple Juice,15513.22,529.0


In [19]:
# Items below reorder point
query("""
    SELECT i.product_id, p.product_name, i.quantity_on_hand, i.reorder_point
    FROM inventory i
    JOIN products p ON i.product_id = p.product_id
    WHERE i.quantity_on_hand < i.reorder_point
    ORDER BY i.quantity_on_hand ASC
    LIMIT 10
""", label="Stock below reorder point (top 10)")


  Stock below reorder point (top 10)
HTTP 200 | status: ok
Rows returned: 10


,product_id,product_name,quantity_on_hand,reorder_point
0,3,FarmHouse Yogurt,0.0,28.0
1,18,MilkyWay Yogurt,0.0,28.0
2,33,GoldenLoaf White Bread,0.0,24.0
3,56,SnackTime Biscuits,0.0,41.0
4,60,PureGlow Shampoo,0.0,16.0
5,108,SeaHarvest Prawns,0.0,14.0
6,25,PrimeChoice Minced Beef,1.0,17.0
7,92,HarmonyCare Shampoo,1.0,16.0
8,109,PrimeChoice Pork Chops,1.0,17.0
9,135,CoolSip Orange Juice,1.0,33.0


,product_id,product_name,quantity_on_hand,reorder_point
0,3,FarmHouse Yogurt,0.0,28.0
1,18,MilkyWay Yogurt,0.0,28.0
2,33,GoldenLoaf White Bread,0.0,24.0
3,56,SnackTime Biscuits,0.0,41.0
4,60,PureGlow Shampoo,0.0,16.0
5,108,SeaHarvest Prawns,0.0,14.0
6,25,PrimeChoice Minced Beef,1.0,17.0
7,92,HarmonyCare Shampoo,1.0,16.0
8,109,PrimeChoice Pork Chops,1.0,17.0
9,135,CoolSip Orange Juice,1.0,33.0


In [20]:
# Revenue per supplier (cross-table join)
query("""
    SELECT s.supplier_name,
           COUNT(DISTINCT p.product_id) AS products,
           ROUND(SUM(si.line_sale_amount), 2) AS total_revenue
    FROM suppliers s
    JOIN products p     ON s.supplier_id = p.supplier_id
    JOIN sales_items si ON p.product_id  = si.product_id
    GROUP BY s.supplier_id
    ORDER BY total_revenue DESC
""", label="Revenue per supplier")


  Revenue per supplier
HTTP 200 | status: ok
Rows returned: 10


,supplier_name,products,total_revenue
0,Evergreen Produce Importers,23,118750.95
1,Lion City Foods Pte Ltd,17,105338.51
2,Pacific Beverages & Co.,14,84469.21
3,Blue Ocean Seafood Traders,13,80676.30
4,Sunrise Dairy Supplies,17,76486.21
5,Prime Choice Meats,13,66886.29
6,Fresh & Crisp Snacks,16,62652.33
7,Golden Harvest Grocers,11,62134.70
8,CleanHome Household Products,15,61589.03
9,Harmony Personal Care,11,55270.11


,supplier_name,products,total_revenue
0,Evergreen Produce Importers,23,118750.95
1,Lion City Foods Pte Ltd,17,105338.51
2,Pacific Beverages & Co.,14,84469.21
3,Blue Ocean Seafood Traders,13,80676.30
4,Sunrise Dairy Supplies,17,76486.21
5,Prime Choice Meats,13,66886.29
6,Fresh & Crisp Snacks,16,62652.33
7,Golden Harvest Grocers,11,62134.70
8,CleanHome Household Products,15,61589.03
9,Harmony Personal Care,11,55270.11


### 3. Blocked Queries (expect HTTP 403 rejected)

In [32]:
# Attempt DROP TABLE — must be rejected
query("DROP TABLE suppliers", label="innocent command")


  innocent command
HTTP 403 | status: rejected
Message: Only SELECT statements are permitted. Data manipulation and schema modification queries are not allowed.


{'status': 'rejected',
 'message': 'Only SELECT statements are permitted. Data manipulation and schema modification queries are not allowed.'}

In [22]:
# Attempt DELETE — must be rejected
query("DELETE FROM products WHERE product_id = 1", label="BLOCKED: DELETE")


  BLOCKED: DELETE
HTTP 403 | status: rejected
Message: Only SELECT statements are permitted. Data manipulation and schema modification queries are not allowed.


{'status': 'rejected',
 'message': 'Only SELECT statements are permitted. Data manipulation and schema modification queries are not allowed.'}

In [23]:
# Attempt UPDATE — must be rejected
query("UPDATE suppliers SET is_active = 0 WHERE supplier_id = 1", label="BLOCKED: UPDATE")


  BLOCKED: UPDATE
HTTP 403 | status: rejected
Message: Only SELECT statements are permitted. Data manipulation and schema modification queries are not allowed.


{'status': 'rejected',
 'message': 'Only SELECT statements are permitted. Data manipulation and schema modification queries are not allowed.'}

In [24]:
# Attempt INSERT — must be rejected
query("INSERT INTO suppliers (supplier_code, supplier_name) VALUES ('X', 'Hacker')", label="BLOCKED: INSERT")


  BLOCKED: INSERT
HTTP 403 | status: rejected
Message: Only SELECT statements are permitted. Data manipulation and schema modification queries are not allowed.


{'status': 'rejected',
 'message': 'Only SELECT statements are permitted. Data manipulation and schema modification queries are not allowed.'}

In [25]:
# Attempt statement stacking via semicolon — DROP is still caught by keyword check
query("SELECT * FROM suppliers; DROP TABLE suppliers", label="BLOCKED: Statement stacking")


  BLOCKED: Statement stacking
HTTP 403 | status: rejected
Message: Query contains a blocked keyword: 'DROP'. Destructive or write operations are not permitted.


{'status': 'rejected',
 'message': "Query contains a blocked keyword: 'DROP'. Destructive or write operations are not permitted."}

In [26]:
# Attempt PRAGMA write — must be rejected
query("PRAGMA foreign_keys = OFF", label="BLOCKED: PRAGMA write")


  BLOCKED: PRAGMA write
HTTP 403 | status: rejected
Message: Only SELECT statements are permitted. Data manipulation and schema modification queries are not allowed.


{'status': 'rejected',
 'message': 'Only SELECT statements are permitted. Data manipulation and schema modification queries are not allowed.'}

### 4. Malformed / Invalid Inputs (expect HTTP 400 or 422 error)

In [27]:
# SELECT from a non-existent table — SQL error
query("SELECT * FROM nonexistent_table", label="ERROR: Non-existent table")


  ERROR: Non-existent table
HTTP 400 | status: error
Message: SQL execution error: no such table: nonexistent_table


{'status': 'error',
 'message': 'SQL execution error: no such table: nonexistent_table'}

In [28]:
# Malformed SQL syntax
query("SELECT FROM WHERE", label="ERROR: Malformed SQL syntax")


  ERROR: Malformed SQL syntax
HTTP 400 | status: error
Message: SQL execution error: near "FROM": syntax error


{'status': 'error',
 'message': 'SQL execution error: near "FROM": syntax error'}

In [29]:
# Empty string input — validation error
query("", label="ERROR: Empty query string")


  ERROR: Empty query string
HTTP 422 | status: error
Message: Invalid input: 'sql' must be a non-empty string.


{'status': 'error',
 'message': "Invalid input: 'sql' must be a non-empty string."}

In [30]:
# Missing 'sql' field entirely — Pydantic 422 validation error
resp = requests.post(f"{BASE_URL}/query", json={"query": "SELECT 1"})
print(f"HTTP {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

HTTP 422
{
  "detail": [
    {
      "type": "missing",
      "loc": [
        "body",
        "sql"
      ],
      "msg": "Field required",
      "input": {
        "query": "SELECT 1"
      }
    }
  ]
}


In [31]:
# SELECT a column that doesn't exist
query("SELECT nonexistent_column FROM suppliers", label="ERROR: Non-existent column")


  ERROR: Non-existent column
HTTP 400 | status: error
Message: SQL execution error: no such column: nonexistent_column


{'status': 'error',
 'message': 'SQL execution error: no such column: nonexistent_column'}